In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm 
from scipy.stats import chi2, chi2_contingency, random_table

file_path = "GenAI_Cleaned_Data.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="Consenting n105"
)

print("Sample size:", len(df))


Sample size: 105


In [3]:
#Job role vs perceived productivity
#Generate the cross-tabulation
role_order = [
    "Analyst / Data-related role",
    "Developer / Programmer",
    "Manager / Lead",
    "Tester / QA",
    "IT Support / Operations"
]

productivity_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

table = pd.crosstab(
    df["role"],
    df["productivity"]
).reindex(
    index=role_order,
    columns=productivity_order,
    fill_value=0
)

print(table)


productivity                 Disagree  Neither agree nor disagree  Agree
role                                                                    
Analyst / Data-related role         1                          11     23
Developer / Programmer              0                           2     22
Manager / Lead                      2                           3     15
Tester / QA                         0                           2     13
IT Support / Operations             0                           6      5


In [5]:
#Calculate the chi-square and Cramer's V 
observed = table.to_numpy()

result = chi2_contingency(
    observed,
    correction=False
)

chi2 = result.statistic
dof = result.dof
expected = result.expected_freq

n = observed.sum()

cramers_v = np.sqrt(
    chi2 / (
        n * min(observed.shape[0] - 1,
                observed.shape[1] - 1)
    )
)

print("Chi-square:", round(chi2, 3))
print("Degrees of freedom:", dof)
print("Cells with expected count < 5:",
      (expected < 5).sum())
print("Cramer's V:", round(cramers_v, 3))


Chi-square: 17.172
Degrees of freedom: 8
Cells with expected count < 5: 8
Cramer's V: 0.286


In [7]:
#Monte Carlo conditional test
rng = np.random.default_rng(20261106)

n_simulations = 200000

simulated_tables = random_table.rvs(
    observed.sum(axis=1),
    observed.sum(axis=0),
    size=n_simulations,
    random_state=rng
)

simulated_chi2 = (
    (simulated_tables - expected) ** 2 / expected
).sum(axis=(1, 2))

extreme_count = np.count_nonzero(
    simulated_chi2 >= chi2 - 1e-12
)

p_monte_carlo = (
    extreme_count + 1
) / (
    n_simulations + 1
)

print(
    "Monte Carlo p-value:",
    round(p_monte_carlo, 4)
)


Monte Carlo p-value: 0.0246


In [9]:
#Frequency of GenAI use vs perceived productivity
#Create the contingency table
frequency_order = [
    "Daily",
    "Weekly",
    "Occasionally",
    "Never"
]

productivity_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

table = pd.crosstab(
    df["use_frequency"],
    df["productivity"]
).reindex(
    index=frequency_order,
    columns=productivity_order,
    fill_value=0
)

print(table)


productivity   Disagree  Neither agree nor disagree  Agree
use_frequency                                             
Daily                 0                          10     51
Weekly                0                           5     18
Occasionally          1                           9      9
Never                 2                           0      0


In [11]:
#Calculate the chi-square statistic and Cramer's V
observed = table.to_numpy()

result = chi2_contingency(
    observed,
    correction=False
)

chi2 = result.statistic
dof = result.dof
expected = result.expected_freq
n = observed.sum()

cramers_v = np.sqrt(
    chi2 / (
        n * min(
            observed.shape[0] - 1,
            observed.shape[1] - 1
        )
    )
)

print("Chi-square:", round(chi2, 3))
print("Degrees of freedom:", dof)
print(
    "Cells with expected count < 5:",
    (expected < 5).sum()
)
print("Cramer's V:", round(cramers_v, 3))


Chi-square: 79.522
Degrees of freedom: 6
Cells with expected count < 5: 7
Cramer's V: 0.615


In [13]:
#Monte Carlo test
rng = np.random.default_rng(20261106)

n_simulations = 200000
extreme_count = 0

# Run in batches to avoid excessive memory use
for i in range(20):

    simulated_tables = random_table.rvs(
        observed.sum(axis=1),
        observed.sum(axis=0),
        size=10000,
        random_state=rng
    )

    simulated_chi2 = (
        (simulated_tables - expected) ** 2
        / expected
    ).sum(axis=(1, 2))

    extreme_count += np.count_nonzero(
        simulated_chi2 >= chi2 - 1e-12
    )

p_monte_carlo = (
    extreme_count + 1
) / (
    n_simulations + 1
)

print("Monte Carlo p-value:", p_monte_carlo)
print("Extreme simulations:", extreme_count)


Monte Carlo p-value: 4.9999750001249995e-06
Extreme simulations: 0


In [15]:
#Frequency vs productivity, excluding never-users – Sensitivity analysis
# Keep only respondents who use GenAI
df_users = df[
    df["use_frequency"] != "Never"
].copy()

print("Sample size:", len(df_users))


Sample size: 103


In [17]:
frequency_order = [
    "Daily", "Weekly", "Occasionally"
]

productivity_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

table_103 = pd.crosstab(
    df_users["use_frequency"],
    df_users["productivity"]
).reindex(
    index=frequency_order,
    columns=productivity_order,
    fill_value=0
)

print(table_103)


productivity   Disagree  Neither agree nor disagree  Agree
use_frequency                                             
Daily                 0                          10     51
Weekly                0                           5     18
Occasionally          1                           9      9


In [19]:
observed_103 = table_103.to_numpy()

result_103 = chi2_contingency(
    observed_103,
    correction=False
)

chi2_103 = result_103.statistic
expected_103 = result_103.expected_freq

cramers_v_103 = np.sqrt(
    chi2_103 / (
        observed_103.sum()
        * min(observed_103.shape[0]-1,
              observed_103.shape[1]-1)
    )
)

print("Chi-square:", round(chi2_103, 3))
print("Degrees of freedom:", result_103.dof)
print("Cramer's V:", round(cramers_v_103, 3))
print(
    "Expected cells below 5:",
    (expected_103 < 5).sum()
)


Chi-square: 12.955
Degrees of freedom: 4
Cramer's V: 0.251
Expected cells below 5: 4


In [21]:
rng = np.random.default_rng(20261106)

n_simulations = 200000
extreme_count = 0

for _ in range(20):
    simulated = random_table.rvs(
        observed_103.sum(axis=1),
        observed_103.sum(axis=0),
        size=10000,
        random_state=rng
    )

    simulated_chi2 = (
        (simulated - expected_103)**2
        / expected_103
    ).sum(axis=(1, 2))

    extreme_count += np.count_nonzero(
        simulated_chi2 >= chi2_103 - 1e-12
    )

p_mc_103 = (extreme_count + 1) / (n_simulations + 1)

print("Monte Carlo p-value:", round(p_mc_103, 4))


Monte Carlo p-value: 0.0056


In [39]:
#Frequency and job role predicting perceived productivity
# Binary Logistic Regression

import statsmodels.api as sm
from scipy.stats import chi2 as chi2_dist

df = pd.read_excel(
    file_path,
    sheet_name="Users n103"
)

print("Sample size:", len(df))


Sample size: 103


In [41]:
# 1 = Agree; 0 = Neutral or Disagree
y = (df["productivity"] == "Agree").astype(int)

print(y.value_counts().sort_index())


productivity
0    25
1    78
Name: count, dtype: int64


In [43]:

X = pd.get_dummies(
    df[["use_frequency", "role"]],
    dtype=float
)

# Set reference categories
X = X.drop(columns=[
    "use_frequency_Occasionally",
    "role_Analyst / Data-related role"
])

X = sm.add_constant(X)


In [45]:

model = sm.Logit(y, X)

result = model.fit(
    method="newton",
    maxiter=500,
    disp=False
)

print(result.summary())
print("Converged:", result.mle_retvals["converged"])


                           Logit Regression Results                           
Dep. Variable:           productivity   No. Observations:                  103
Model:                          Logit   Df Residuals:                       96
Method:                           MLE   Df Model:                            6
Date:                Sun, 15 Nov 2026   Pseudo R-squ.:                  0.1494
Time:                        06:21:15   Log-Likelihood:                -48.556
converged:                       True   LL-Null:                       -57.082
Covariance Type:            nonrobust   LLR p-value:                  0.009092
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           -0.4356      0.625     -0.697      0.486      -1.661       0.790
use_frequency_Daily              1.5200      0.624      2.435      0.015     

In [47]:

ci = result.conf_int()

regression_table = pd.DataFrame({
    "Odds Ratio": np.exp(result.params),
    "CI Lower": np.exp(ci[0]),
    "CI Upper": np.exp(ci[1]),
    "p-value": result.pvalues
})

# Exclude the intercept
regression_table = regression_table.drop("const")

print(regression_table.round(3))


                              Odds Ratio  CI Lower  CI Upper  p-value
use_frequency_Daily                4.572     1.345    15.542    0.015
use_frequency_Weekly               3.234     0.727    14.386    0.123
role_Developer / Programmer        5.384     1.022    28.365    0.047
role_IT Support / Operations       0.654     0.142     3.020    0.587
role_Manager / Lead                2.007     0.497     8.094    0.328
role_Tester / QA                   3.020     0.550    16.577    0.203


In [49]:
#Likelihood ratio test 
#Overall contribution of usage frequency and job role

frequency_cols = [
    c for c in X.columns
    if c.startswith("use_frequency_")
]

role_cols = [
    c for c in X.columns
    if c.startswith("role_")
]

for group_name, cols in [
    ("Usage frequency", frequency_cols),
    ("Job role", role_cols)
]:

    reduced_X = X.drop(columns=cols)

    reduced_result = sm.Logit(
        y, reduced_X
    ).fit(disp=False)

    lr_stat = 2 * (
        result.llf - reduced_result.llf
    )

    df = len(cols)

    p_value = chi2_dist.sf(lr_stat, df)

    print(
        group_name,
        "LR chi-square:", round(lr_stat, 3),
        "df:", df,
        "p-value:", round(p_value, 4)
    )

Usage frequency LR chi-square: 5.958 df: 2 p-value: 0.0508
Job role LR chi-square: 7.689 df: 4 p-value: 0.1036
